# 🐱🎙️ KittenASR-Go · Quick Test on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itamaker/kitten-asr-go/blob/main/examples/kitten-asr-go-colab.ipynb)

Try [`itamaker/kitten-asr-go`](https://github.com/itamaker/kitten-asr-go) on Colab (Ubuntu / x86_64, **no GPU needed**) — a pure-Go speech-to-text engine for KittenML's `kitten-asr-tiny` / `kitten-asr-small-enhanced` models.

## ① Setup

Install dependencies (Go, ONNX Runtime, ffmpeg for building a test clip, and a C compiler — the ONNX Runtime binding's dlopen shim is cgo, even though ONNX Runtime itself is dlopen'd at runtime rather than linked) and clone the repo. ONNX Runtime is pinned to 1.30.0 to satisfy the Go bindings' minimum C API version — anything older than ONNX Runtime 1.24 fails to load with an explicit API-version error rather than silently misbehaving.

In [ ]:
%env ONNXRUNTIME_LIB_PATH=/usr/local/lib/libonnxruntime.so.1.30.0

!sudo apt-get update -qq && sudo apt-get install -y -qq build-essential ffmpeg
!wget -q https://go.dev/dl/go1.26.0.linux-amd64.tar.gz && sudo rm -rf /usr/local/go && sudo tar -C /usr/local -xzf go1.26.0.linux-amd64.tar.gz && sudo ln -sf /usr/local/go/bin/go /usr/local/bin/go
!wget -q https://github.com/microsoft/onnxruntime/releases/download/v1.30.0/onnxruntime-linux-x64-1.30.0.tgz && tar xzf onnxruntime-linux-x64-1.30.0.tgz && sudo cp onnxruntime-linux-x64-1.30.0/lib/libonnxruntime.so* /usr/local/lib/ && sudo ldconfig
!go version

In [ ]:
!git clone -q https://github.com/itamaker/kitten-asr-go.git
%cd kitten-asr-go
!go build -o bin/ ./...

## ② Get the model

Downloads `kitten-asr-tiny`'s ONNX export (~1.8 GB — this'll take a few minutes) via `scripts/fetch_model.sh`. Swap `tiny` for `small-enhanced` to try the bigger model instead (~3.7 GB).

In [ ]:
!chmod +x scripts/fetch_model.sh
!scripts/fetch_model.sh tiny

## ③ Transcribe

No sample audio handy in a fresh Colab runtime, so this synthesizes a short test clip with [`edge-tts`](https://github.com/rany2/edge-tts) first — a free wrapper around Microsoft Edge's neural voices, much closer to natural speech than a formant synthesizer like `espeak-ng` (whose robotic output is out-of-distribution for the model and won't transcribe as cleanly as real speech). Swap in your own `.wav` (upload via the Colab file browser, then point `AUDIO` at it) to try real speech instead.

In [ ]:
!pip install -q edge-tts
!edge-tts --voice en-US-AriaNeural --text "Hello, this is a test of the kitten automatic speech recognition system." --write-media /tmp/hello_tts.mp3
# edge-tts writes mp3-encoded audio regardless of the output extension --
# decode it to a real PCM .wav so every step below can just use .wav.
!ffmpeg -y -loglevel error -i /tmp/hello_tts.mp3 -ar 16000 -ac 1 /content/hello.wav

from IPython.display import Audio
Audio('/content/hello.wav')

In [ ]:
AUDIO = '/content/hello.wav'
!./bin/kitten-asr -language ./models/kitten-asr-tiny-onnx {AUDIO}

**(Optional) OpenAI-compatible API server** — two steps: **Step 1** starts the server on port 8888 (`nohup … &`), **Step 2** uploads the clip via `curl`. Edit `response_format` (`json` / `text` / `verbose_json`) and re-run Step 2 as much as you like.

In [ ]:
# Step 1 — start the server on port 8888
!nohup ./bin/kitten-asr-server -host 127.0.0.1 -port 8888 ./models/kitten-asr-tiny-onnx > /content/server.log 2>&1 &
!sleep 20; cat /content/server.log

In [ ]:
# Step 2 — transcribe via the API (edit response_format and re-run as you like)
!curl -sS http://127.0.0.1:8888/v1/audio/transcriptions \
    -F "file=@/content/hello.wav" \
    -F "response_format=verbose_json"

**(Optional) Streaming API test** — the server also exposes a WebSocket endpoint, `/v1/audio/transcriptions/stream`, for live audio instead of a single upload. It's windowed re-transcription on a timer (not frame-by-frame causal decoding — see `asr.Stream`'s doc comment in the repo), so expect a `partial` message every couple of seconds as audio streams in, revised as more context arrives, and a `final` message once the client sends `end`. This reuses the server already started above, so run that cell first.

In [ ]:
!pip install -q websockets

import asyncio, json, wave
import websockets

async def stream_transcribe(path, host='127.0.0.1', port=8888, chunk_ms=200):
    with wave.open(path, 'rb') as wf:
        sample_rate = wf.getframerate()
        sample_width = wf.getsampwidth()
        channels = wf.getnchannels()
        pcm = wf.readframes(wf.getnframes())

    url = (f'ws://{host}:{port}/v1/audio/transcriptions/stream'
           f'?sample_rate={sample_rate}&bits_per_sample={sample_width * 8}&channels={channels}')
    chunk_bytes = int(sample_rate * sample_width * channels * chunk_ms / 1000)

    # ping_interval=None: CPU-only inference (no GPU on this Colab runtime) can
    # make a single Feed() pass take longer than the client's default 20s ping
    # timeout, and the server's read loop can't answer a ping while it's busy
    # inside a Feed() call -- so the default keepalive gives up on a server
    # that's just slow, not dead. This is a short-lived demo connection, not a
    # long-idle one, so skipping client-side keepalive pings is safe here.
    async with websockets.connect(url, ping_interval=None) as ws:
        async def receiver():
            async for raw in ws:
                msg = json.loads(raw)
                print(f"[{msg['type']}] {msg.get('text', msg.get('message', ''))}")
                if msg['type'] == 'final':
                    return

        recv_task = asyncio.create_task(receiver())
        for i in range(0, len(pcm), chunk_bytes):
            await ws.send(pcm[i:i + chunk_bytes])
            await asyncio.sleep(chunk_ms / 1000)  # simulate real-time playback
        await ws.send(json.dumps({'type': 'end'}))
        await recv_task

await stream_transcribe('/content/hello.wav')